# Battery Feature Lab: Catenaro–Onori example

Read one raw file with BDS, run BFL, inspect the compact result, retrieve core evidence, and optionally ask GPT-5.6 to interpret the saved JSON records.

Data: Catenaro, Edoardo; Onori, Simona (2021), *Experimental data of three lithium-ion batteries under galvanostatic discharge tests at different C-rates and operating temperatures*, Version 2, [doi:10.17632/kxsbr4x3j2.2](https://doi.org/10.17632/kxsbr4x3j2.2), CC BY 4.0.

Install the optional API client with `uv sync --extra ai`. The API cell reads `OPENAI_API_KEY` from the environment or requests it with a hidden prompt; the key is never written into this notebook.

## 1. Set up the example

Resolve repository-relative paths and show the installed BDS and BFL versions.

In [ ]:
import importlib.metadata
import json
from pathlib import Path

import bds

import bfl

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").is_file() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
dataset_path = Path("examples/data/Catenaro_Onori_2021/NCA_k1_0_05C_05degC.xlsx")
input_path = repo_root / dataset_path
output_path = (
    Path("examples/outputs/Catenaro_Onori_2021") / input_path.stem
)
if not input_path.is_file():
    raise FileNotFoundError(input_path)
{
    "input_file": dataset_path.as_posix(),
    "output_directory": output_path.as_posix(),
    "battery-data-standard": importlib.metadata.version("battery-data-standard"),
    "battery-feature-lab": importlib.metadata.version("battery-feature-lab"),
}

## 2. Preprocess the raw measurements with BDS

Read the workbook without interpolation or silent repair and inspect the conversion report.

In [ ]:
bds_frame, bds_report = bds.read_with_report(
    input_path,
    cycler="auto",
    strict=False,
    keep_raw=True,
    current_sign="charge-positive",
    repair_policy="warn",
    time_sampling_policy="warn",
    current_sign_check="none",
)
{
    "rows": bds_frame.height,
    "columns": bds_frame.columns,
    "cycler": bds_report.to_dict()["cycler"],
    "unmapped_columns": bds_report.to_dict()["unmapped_columns"],
    "time_sampling": bds_report.to_dict()["metadata"]["time_sampling"],
    "semantic_sources": bds_report.to_dict()["metadata"]["semantic_sources"],
}

## 3. Analyse the standardised measurements with BFL

Generate the machine-readable analysis, metadata, evidence, and validation artifacts.

In [ ]:
result = bfl.analyze(
    input_path,
    output_dir=repo_root / output_path,
    input_adapter="bds",
    temperature_column="raw:Surface_Temp(degC)",
)
[path.name for path in result.files]

## 4. Inspect the compact analysis result

Use the compact JSON as the discovery index for downstream tools.

In [ ]:
analysis = json.loads(result.analysis_results_path.read_text(encoding="utf-8"))
metadata = json.loads(result.analysis_metadata_path.read_text(encoding="utf-8"))
validation = json.loads(result.analysis_validation_path.read_text(encoding="utf-8"))
operation_index = analysis["dimensions"]["operation"][0]
{
    "validation_status": validation["status"],
    "output_files": [path.name for path in result.files],
    "output_sizes_bytes": {path.name: path.stat().st_size for path in result.files},
    "dimensions": {
        name: [item["record_type"] for item in items]
        for name, items in analysis["dimensions"].items()
    },
    "operation_sequence": operation_index["attributes"]["operation_sequence"],
    "operation_metrics": operation_index["metrics"],
    "metadata_channels": metadata["channels"],
    "short_window_recomputation": validation["recomputation"]["short_window_recomputation"],
}

## 5. Retrieve detailed evidence

Follow record identifiers from the compact result to source intervals, methods, and quality limits.

In [ ]:
evidence = json.loads(result.analysis_evidence_path.read_text(encoding="utf-8"))
core_types = {
    "response.capacity_aligned_profile",
    "response.current_step_summary",
    "response.relaxation_signature",
}
core_index = {
    item["record_type"]: item
    for item in analysis["dimensions"]["response"]
    if item["record_type"] in core_types
}
core_evidence = {
    record_type: next(
        item
        for item in evidence["records"]
        if item["record_id"] == index["evidence"]["record_id"]
    )
    for record_type, index in core_index.items()
}
{
    record_type: {
        "compact_result": core_index[record_type],
        "source_intervals": record["source_intervals"],
        "method": record["method"],
    }
    for record_type, record in core_evidence.items()
}

## 6. Prepare the controlled AI inputs

Keep the first three conditions as original files and configure one shared GPT-5.6 prompt without a token-length setting.

In [ ]:
import os
from getpass import getpass

from openai import OpenAI

bds_json_path = result.input_report_path
bds_bfl_json_paths = (
    result.input_report_path,
    result.analysis_metadata_path,
    result.analysis_results_path,
    result.analysis_evidence_path,
    result.analysis_validation_path,
)

prompt = """ 
You are a battery engineer. 
 
Analyse the supplied battery data and explain: 
1. what the battery experienced, 
2. how it responded, 
3. how it evolved. 
   
"""

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
client = OpenAI()
response_settings = {
    "model": "gpt-5.6",
    "reasoning": {"effort": "medium"},
    "text": {"verbosity": "medium"},
    "store": False,
}

def upload_original_files(paths):
    uploaded = []
    for path in paths:
        with path.open("rb") as source:
            uploaded.append(client.files.create(file=source, purpose="user_data"))
    return uploaded

def original_file_condition(uploaded_files):
    return [
        {
            "role": "user",
            "content": [
                *[
                    {"type": "input_file", "file_id": item.id}
                    for item in uploaded_files
                ],
                {"type": "input_text", "text": prompt},
            ],
        }
    ]

def delete_uploaded_files(uploaded_files):
    for item in uploaded_files:
        client.files.delete(item.id)

{
    "raw_input": dataset_path.as_posix(),
    "bds_only": [bds_json_path.name],
    "bds_bfl": [path.name for path in bds_bfl_json_paths],
}

## 7. Experiment 1 — raw measurements

Upload the unchanged Excel workbook directly. OpenAI currently parses up to the first 1,000 rows per sheet for spreadsheet `input_file` requests; the temporary Files API object is deleted after the response.

In [ ]:
uploaded_raw_files = upload_original_files((input_path,))
try:
    raw_response = client.responses.create(
        **response_settings,
        input=original_file_condition(uploaded_raw_files),
    )
finally:
    delete_uploaded_files(uploaded_raw_files)
print(raw_response.output_text)

## 8. Experiment 2 — BDS JSON only

Upload only the unchanged BDS conversion report to test what preprocessing metadata can support without the measurements.

In [ ]:
uploaded_bds_files = upload_original_files((bds_json_path,))
try:
    bds_only_response = client.responses.create(
        **response_settings,
        input=original_file_condition(uploaded_bds_files),
    )
finally:
    delete_uploaded_files(uploaded_bds_files)
print(bds_only_response.output_text)

## 9. Experiment 3 — BDS + BFL JSON

Upload the unchanged BDS report together with every BFL JSON artifact as the complete machine-readable condition.

In [ ]:
uploaded_bds_bfl_files = upload_original_files(bds_bfl_json_paths)
try:
    bds_bfl_response = client.responses.create(
        **response_settings,
        input=original_file_condition(uploaded_bds_bfl_files),
    )
finally:
    delete_uploaded_files(uploaded_bds_bfl_files)
print(bds_bfl_response.output_text)

## 10. Experiment 4 — BFL-guided evidence retrieval

Use compact-result evidence identifiers to retrieve the records relevant to Operation, Response, and Evolution before sending a smaller JSON condition.

In [ ]:
referenced_record_ids = {
    item["evidence"]["record_id"]
    for items in analysis["dimensions"].values()
    for item in items
    if item.get("evidence", {}).get("record_id")
}
retrieved_condition = {
    result.input_report_path.name: json.loads(
        result.input_report_path.read_text(encoding="utf-8")
    ),
    result.analysis_metadata_path.name: metadata,
    result.analysis_results_path.name: analysis,
    "retrieved_evidence_records": [
        item for item in evidence["records"] if item["record_id"] in referenced_record_ids
    ],
    result.analysis_validation_path.name: validation,
}
retrieval_response = client.responses.create(
    **response_settings,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {
                    "type": "input_text",
                    "text": json.dumps(retrieved_condition, ensure_ascii=False),
                },
            ],
        }
    ],
)
print(retrieval_response.output_text)